# GRPO reward-scaling experiment: Colab A100 runner

One-click pipeline for the AISTATS LLM experiment: Qwen2.5-0.5B-Instruct + LoRA, TRL
`GRPOTrainer`, comparing GRPO / Dr.GRPO / global normalization reward scaling under a
**conflicting-preference task**. Every training example asks the SAME question ("pick
an integer between 0 and 100"), but carries a hidden family label (never shown to the
model): family A wants the answer near 90, family B wants it near 50, and B's WHOLE
reward is scaled by `k`. A and B pull the SAME output distribution in opposite
directions, so `k` should visibly shift where the policy settles -- unless an
estimator's own reward normalization cancels that shift out. v3: both families' base
rewards are also scaled by 100 (`config.REWARD_SCALE`), which keeps TRL's hardcoded
+1e-4 std-normalization epsilon negligible even once the policy converges near a
target and natural reward variance shrinks -- without it, that epsilon alone can
reintroduce a small k-dependent bias into GRPO's otherwise-exact invariance.

**Run cells top to bottom.** There is no separately gated pilot step: all 7 configs
run directly, seed 0 first, with a pass/fail check printed right after seed 0 so you
can stop early if the design isn't working before spending seeds 1-2's GPU time.
Target: ~2 hours total on one A100 (short, 8-token completions; 120 steps/run).

All results are saved to Google Drive (`grpo_llm_results/`) immediately after each
run finishes, so the sweep can resume from a Colab disconnect without losing completed
work.

## 1. Check for a GPU

Sets `PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True` before torch ever loads --
it reduces allocator fragmentation under GRPO's bursty generate-then-backward memory
pattern, and only takes effect if set prior to CUDA context init.

In [ ]:
import os

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import subprocess

try:
    import torch
    has_gpu = torch.cuda.is_available()
except ImportError:
    has_gpu = False

if not has_gpu:
    raise RuntimeError(
        'No GPU detected. Go to Runtime > Change runtime type, select a GPU '
        '(A100 recommended), then Runtime > Restart session and run all cells again.'
    )

print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)

## 2. Config -- edit if your fork/repo name differs

In [ ]:
GITHUB_USER = "nilay47"
GITHUB_REPO = "agenthon"
DRIVE_RESULTS_DIR = "/content/drive/MyDrive/grpo_llm_results"
REPO_DIR = f"/content/{GITHUB_REPO}"

## 3. Mount Google Drive

Also points the HuggingFace cache at Drive, so the ~1GB model download survives a
disconnect instead of being wiped along with the rest of the Colab VM's disk.

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')
os.makedirs(DRIVE_RESULTS_DIR, exist_ok=True)
print(f"Results will be saved to {DRIVE_RESULTS_DIR}")

os.environ["HF_HOME"] = "/content/drive/MyDrive/hf_cache"
os.makedirs(os.environ["HF_HOME"], exist_ok=True)
hf_home = os.environ['HF_HOME']
print(f"HF cache: {hf_home}")

## 4. Clone the repo

If the repo is **private**, add a `GITHUB_TOKEN` Colab secret first (key icon in the
left sidebar) with a personal access token that has `repo` scope, and grant this
notebook access to it when prompted. If the repo is public, this works with no token.

In [ ]:
import os

token = None
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    pass

if token:
    clone_url = f"https://{token}@github.com/{GITHUB_USER}/{GITHUB_REPO}.git"
else:
    clone_url = f"https://github.com/{GITHUB_USER}/{GITHUB_REPO}.git"

if not os.path.exists(REPO_DIR):
    !git clone {clone_url} {REPO_DIR}
else:
    print(f"{REPO_DIR} already exists, skipping clone (pulling latest instead)")
    !cd {REPO_DIR} && git pull

%cd {REPO_DIR}/llm

## 5. Install dependencies (Colab-safe: never touches torch/CUDA)

Installs `trl`/`peft`/`transformers`/`accelerate`/`datasets` with `--no-deps`, so pip
never resolves THEIR torch dependency against Colab's preinstalled build -- that
resolution is what previously pulled in a mismatched torch/CUDA wheel and broke
`trl.trainer.grpo_trainer` with `libcudart.so.13: cannot open shared object file`.
The lightweight, pure-Python packages those five need are installed separately with
normal resolution (they can't touch torch/CUDA either way). vLLM is intentionally
**not** installed -- `run.py` already falls back to plain HF generation automatically.

Colab also preinstalls an old `torchao` that `peft` refuses to import against
(`Found an incompatible version of torchao`) -- we never use it, so it's removed.

In [ ]:
!pip install -q --no-deps -r requirements-colab.txt
!pip install -q huggingface_hub tokenizers safetensors regex pyyaml filelock requests \
    tqdm packaging psutil pandas pyarrow dill multiprocess fsspec xxhash scipy
!pip uninstall -y -q torchao

import importlib.metadata as _metadata

for _pkg in ['torch', 'transformers', 'trl', 'peft', 'accelerate', 'datasets']:
    try:
        _ver = _metadata.version(_pkg)
    except _metadata.PackageNotFoundError:
        _ver = 'NOT INSTALLED'
    print(f"{_pkg} == {_ver}")

import torch

print(f"torch.__version__ = {torch.__version__}")
print(f"torch.version.cuda = {torch.version.cuda}")
print(f"torch.cuda.is_available() = {torch.cuda.is_available()}")

try:
    import trl.trainer.grpo_trainer  # noqa: F401
    from peft import LoraConfig, get_peft_model  # noqa: F401
    print("trl.trainer.grpo_trainer and peft imported OK")
except Exception as e:
    raise RuntimeError(
        f"Import check failed against torch {torch.__version__} (CUDA {torch.version.cuda}). "
        f"This usually means a pip install pulled in a mismatched torch/CUDA build, or an "
        f"incompatible optional dependency (like torchao) is still installed. "
        f"Original error: {e!r}"
    ) from e

## 5b. PREFLIGHT: exercise every training config on 2 tiny steps

Loads the real model + LoRA and runs 2 GRPO steps (2 prompts x G=4, 8-token
completions -- the real `max_completion_length`) for **each** of the four training
configs used later -- `scale_rewards` `"group"`, `"batch"`, `"none"`, and the
sigma-sampling method -- including one eval-callback call and one save to the Drive
`preflight/` subfolder each. This is the exact same code path `run.py` uses for the
real runs, just at a scale that finishes in seconds, so a config-level break (like the
torchao import failure above) surfaces here instead of after committing GPU time.

In [ ]:
import os
import sys
sys.path.insert(0, '.')
import config
from run import run_one

PREFLIGHT_DIR = os.path.join(DRIVE_RESULTS_DIR, 'preflight')

for _method in ['grpo', 'global', 'drgrpo', 'sigma_sampling']:
    print(f'--- preflight: {_method} ---')
    try:
        run_one(_method, k=1, seed=0, max_steps=2, out_dir=PREFLIGHT_DIR,
                prompts_per_step=2, g=4, max_completion_length=config.MAX_COMPLETION_LENGTH,
                eval_every=1)
    except Exception as e:
        raise RuntimeError(
            f"PREFLIGHT FAILED for config '{_method}'. Original error: {e!r}"
        ) from e

print("PREFLIGHT PASSED")

## 5c. PREFLIGHT: peak GPU memory at the REAL batch shape

The config-smoke preflight above uses a tiny 2-prompt batch, which does not stress
memory. This one runs the **actual** training shape (16 prompts x G=8 = 128
completions per optimizer step, full batch in one backward -- no micro-batching, since
8-token completions keep `grad_logits` small enough not to need it) for one config,
with completions forced out to the full `max_completion_length` via `min_new_tokens`
(the true worst case), and reports peak GPU memory. Should now trivially pass (expect
well under 2 GB) -- kept as a safety net in case `config.MAX_COMPLETION_LENGTH` or
`config.MICRO_BATCH_COMPLETIONS` is ever changed back toward the old, longer-completion
regime. Fails if peak reserved memory exceeds 34 GB (A100 is 40GB).

In [ ]:
import torch

torch.cuda.reset_peak_memory_stats()
run_one('grpo', k=1, seed=0, max_steps=1, out_dir=PREFLIGHT_DIR,
        prompts_per_step=16, g=8, max_completion_length=config.MAX_COMPLETION_LENGTH, eval_every=1,
        extra_grpo_kwargs=dict(generation_kwargs=dict(min_new_tokens=config.MAX_COMPLETION_LENGTH)))

peak_allocated_gb = torch.cuda.max_memory_allocated() / 1024**3
peak_reserved_gb = torch.cuda.max_memory_reserved() / 1024**3
print(f"peak memory allocated: {peak_allocated_gb:.2f} GB")
print(f"peak memory reserved:  {peak_reserved_gb:.2f} GB")

if peak_reserved_gb > 34:
    current_micro = config.MICRO_BATCH_COMPLETIONS
    raise RuntimeError(
        f"Peak GPU memory ({peak_reserved_gb:.2f} GB reserved) exceeds the 34 GB safety "
        f"threshold for a 40GB A100. Reduce config.MICRO_BATCH_COMPLETIONS (currently "
        f"{current_micro}) and rerun this cell."
    )
print("MEMORY PREFLIGHT PASSED")

## 6. Baseline sanity check

Runs the **untrained** model on 100 fresh prompts: reports the initial distribution
of the parsed integer u (mean, median, 10-bin histogram over [0,100]) and parse rate,
plus an informational within-group-std-of-u check at training temperature (not a hard
gate -- just a flag if there's too little output variety for GRPO to have any signal).

In [ ]:
!python baseline_eval.py --out_dir "{DRIVE_RESULTS_DIR}"

import json, os
with open(os.path.join(DRIVE_RESULTS_DIR, "baseline_eval.json")) as f:
    baseline_result = json.load(f)
print(baseline_result)

## 7. Determinism check (GRPO advantages must be k-invariant)

GRPO's per-group reward normalization, `(r - mean(r)) / (std(r) + eps)`, is an exact
invariant of a PURE rescaling `r -> k*r` as eps becomes negligible -- the mechanism
behind GRPO being predicted to erase k's effect. This runs GRPO k=1 and GRPO k=10 for
ONE step each, same seed (`run_one` reseeds torch before building the model, so both
calls generate identical completions), and checks the first step's advantages match
to a RELATIVE tolerance of 1e-4 (not absolute -- v3's `REWARD_SCALE=100` is specifically
what keeps TRL's hardcoded eps negligible here). Quick (two 1-step runs) -- confirms
the mechanism before spending GPU time on full reruns below.

In [ ]:
import sys
sys.path.insert(0, '.')
from check_determinism import run_check

determinism_passed, determinism_rel_diff = run_check(seed=0)

## 8. Wipe and rerun the full sweep (v3: new targets, reward scale, 120 steps)

v3 moves both targets inward (A:90, B:50, from 80/20) and scales both families' base
rewards by 100 (see cell 1's note on why) -- on top of an earlier bug fix (family B's
unparsable penalty now correctly scales by k instead of staying fixed at -1) and a
seeding fix (`run_one` now reseeds torch before building the model, so repeated calls
with the same seed are actually reproducible). Rather than reasoning about which old
results any of this did or didn't invalidate, this deletes EVERY existing
`<method>_k<k>_seed<seed>.json` in `DRIVE_RESULTS_DIR` (all 7 configs x all 3 seeds)
and reruns the full sweep from scratch, so every reported number comes from the same,
current code.

In [ ]:
import os
from full import CONFIGS, run_sweep, result_path

for _method, _k in CONFIGS:
    for _seed in [0, 1, 2]:
        _path = result_path(DRIVE_RESULTS_DIR, _method, _k, _seed)
        if os.path.exists(_path):
            print('deleting (pre-fix):', _path)
            os.remove(_path)

all_results = run_sweep([0, 1, 2], DRIVE_RESULTS_DIR, skip_existing=True)

## 9. Seed-aware pass check

On the primary metric (final SAMPLED mean u, pooled across families):
(1) GRPO: k=10's mean must fall within k=1's OWN 95% seed-to-seed CI -- indistinguishable
from k=1's natural seed variability. (2) Dr.GRPO and (3) Global: each one's paired
(per-seed) k=1-minus-k=10 shift must have a 95% CI that excludes zero -- a real,
seed-robust shift, not one lucky seed (Global is checked the same way as Dr.GRPO here:
its POOLED, whole-batch normalization doesn't have GRPO's per-group exact-cancellation
property, so it's predicted to shift too). (4) GRPO+sigma-sampling (k=10 only): its mean
must fall within Dr.GRPO k=10's CI -- the dynamic family-oversampling is predicted to
achieve a shift comparable to Dr.GRPO's unnormalized one.

In [ ]:
import json, os
from pilot import check_pass

results_for_check = {
    key: [all_results[f'{key}_seed{s}'] for s in [0, 1, 2]]
    for key in ['grpo_k1', 'grpo_k10', 'drgrpo_k1', 'drgrpo_k10',
                'global_k1', 'global_k10', 'sigma_sampling_k10']
}
passed, detail, final_u = check_pass(results_for_check)

print(detail)
print('SEED-AWARE CHECK', 'PASSED' if passed else 'FAILED')

with open(os.path.join(DRIVE_RESULTS_DIR, "pilot_summary.json"), "w") as f:
    json.dump(dict(passed=passed, detail=detail, final_mean_u=final_u,
                   determinism_passed=determinism_passed,
                   determinism_rel_diff=determinism_rel_diff), f, indent=2)

## 10. Final summary: table + figure

Table: final (pooled) SAMPLED mean u per config, mean +- 95% CI across the 3 seeds.
Figure: mean u over training steps, one panel per method that has both k values
(Dr.GRPO / GRPO / Global norm) -- k=1 solid, k=10 dashed -- with dotted horizontal
lines at the predicted mean u (GRPO ~70 for both k; Dr.GRPO and Global ~70 at k=1,
~54 at k=10). GRPO+sigma-sampling (k=10 only) appears in the table but not the figure.

In [ ]:
import os
from full import build_summary
from summarize import print_summary_table, make_figure

summary = build_summary(all_results, [0, 1, 2])
print_summary_table(summary)
fig_paths = make_figure(summary, os.path.join(DRIVE_RESULTS_DIR, 'summary_figure'))
print("wrote", fig_paths)

import json
with open(os.path.join(DRIVE_RESULTS_DIR, "full_summary.json"), "w") as f:
    json.dump(summary, f, indent=2)

## 11. k-sweep: add k=2, k=5 for GRPO/Dr.GRPO/Global

Runs ONLY the missing `(method, k, seed)` combos -- `skip_existing=True` with no
deletion step, so the existing k=1/k=10 results (and everything else) are left alone.
18 new short runs (3 methods x 2 k-values x 3 seeds), saved to Drive as each finishes.

In [ ]:
from k_sweep import K_SWEEP_NEW_CONFIGS

k_sweep_new_results = run_sweep([0, 1, 2], DRIVE_RESULTS_DIR, skip_existing=True,
                                 configs=K_SWEEP_NEW_CONFIGS)

## 12. k-sweep table + figure

Final SAMPLED mean u (mean +- 95% CI over 3 seeds) vs k in {1,2,5,10}, one marker
style per method, plus the analytic curve `u*(k) = (c_A + c_B*k)/(1+k)` (the
weighted-centroid prediction for a method whose k-scaling directly adds gradient
weight to family B, i.e. one WITHOUT GRPO's per-group cancellation) and a flat
reference line at GRPO's own empirical k=1 mean. Reads k=1/k=10 from `all_results`
(section 8) and k=2/k=5 from the Drive files just written above, so it works
whether or not this cell runs in the same session as the main sweep.

In [ ]:
import os
from k_sweep import load_k_sweep_results, build_k_sweep_table, print_k_sweep_table, make_k_sweep_figure

k_sweep_results = load_k_sweep_results(DRIVE_RESULTS_DIR)
k_sweep_table = build_k_sweep_table(k_sweep_results)
print_k_sweep_table(k_sweep_table)
k_sweep_fig_paths = make_k_sweep_figure(k_sweep_table, os.path.join(DRIVE_RESULTS_DIR, 'k_sweep'))
print("wrote", k_sweep_fig_paths)

## 13. Final data: TRL-epsilon ablation + k=1,2,5,10 paired report

Two reportable items for the paper, no new infrastructure:

1. **TRL epsilon ablation**: GRPO with `REWARD_SCALE=1` (no x100 rescaling) at k=1 and
   k=10, 3 seeds each, otherwise identical to v3 -- a direct empirical test of how much
   the x100 rescaling matters (TRL hardcodes `eps=1e-4` inside GRPO's group-std
   normalization; v3's design rests on REWARD_SCALE=100 keeping that negligible).
   Results are saved to `DRIVE_RESULTS_DIR/eps1/` -- a separate subfolder, so they
   never collide with the existing `grpo_k{k}_seed{s}.json` (REWARD_SCALE=100) files --
   and `config.REWARD_SCALE` is restored to 100 afterward regardless of outcome, so any
   cell run after this one is unaffected. ~6 short runs, well under 15 minutes.
2. **Per-seed k=1,2,5,10 paired report** for GRPO, reusing results already on disk from
   the main sweep (section 8) and the k-sweep (section 11) -- no new training.

Both reports print a table and save to
`DRIVE_RESULTS_DIR/final_eps_ablation.json` /
`DRIVE_RESULTS_DIR/final_grpo_k_sweep_paired.json` (skipped with a message if either
file already exists, so re-running this cell never clobbers a prior result).

In [ ]:
import json, os
from eps_ablation import run_eps_ablation, build_eps_ablation_report, print_eps_ablation_report
from k_sweep import load_k_sweep_results, build_paired_vs_k1_table, print_paired_vs_k1_table

# --- item 1: REWARD_SCALE=1 ablation ---
eps1_results = run_eps_ablation(DRIVE_RESULTS_DIR, skip_existing=True)
eps_report = build_eps_ablation_report(DRIVE_RESULTS_DIR)
print_eps_ablation_report(eps_report)

eps_report_path = os.path.join(DRIVE_RESULTS_DIR, 'final_eps_ablation.json')
if os.path.exists(eps_report_path):
    print(f"\nSKIPPED saving {eps_report_path} (already exists -- not overwriting)")
else:
    with open(eps_report_path, 'w') as f:
        json.dump(eps_report, f, indent=2)
    print(f"\nwrote {eps_report_path}")

# --- item 2: per-seed k=1,2,5,10 paired-vs-k1 report (existing results only) ---
k_sweep_all_results = load_k_sweep_results(DRIVE_RESULTS_DIR)
paired_k_table = build_paired_vs_k1_table(k_sweep_all_results)
print_paired_vs_k1_table(paired_k_table)

paired_k_path = os.path.join(DRIVE_RESULTS_DIR, 'final_grpo_k_sweep_paired.json')
if os.path.exists(paired_k_path):
    print(f"\nSKIPPED saving {paired_k_path} (already exists -- not overwriting)")
else:
    with open(paired_k_path, 'w') as f:
        json.dump(paired_k_table, f, indent=2)
    print(f"\nwrote {paired_k_path}")